# Simple XGBoost Regression – Eye Medical Dataset

### Project
**Predict Intraocular Pressure (IOP) in mmHg using eye and patient features**

This notebook is designed for beginners and works well in **Google Colab**.

> **Important:** The included dataset is synthetic and is for teaching/demo purposes only.  
> It must **not** be used for diagnosis or clinical decision-making.

### Features
- Age
- Corneal thickness
- Cup-to-disc ratio
- Visual field mean deviation
- Blood pressure
- Corneal curvature
- Family history of glaucoma

### Target
- `intraocular_pressure_mmhg`


## 1. Install XGBoost

In [ ]:
!pip -q install xgboost

## 2. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from xgboost import XGBRegressor

print("Libraries imported successfully.")

## 3. Load Dataset

### Google Colab option
Upload `eye_iop_regression_dataset.csv` using the **Files** panel on the left, then run this cell.

If the CSV is not found, the notebook will generate the same type of synthetic eye dataset automatically.


In [ ]:
import os

file_name = "eye_iop_regression_dataset.csv"

if os.path.exists(file_name):
    df = pd.read_csv(file_name)
    print("CSV dataset loaded.")
else:
    print("CSV not found, so a synthetic eye dataset will be generated.")

    rng = np.random.default_rng(42)
    n = 600

    age = rng.integers(18, 81, n)
    corneal_thickness_um = np.clip(rng.normal(540, 35, n), 440, 650)
    cup_disc_ratio = np.clip(rng.normal(0.45, 0.14, n), 0.15, 0.90)
    visual_field_md_db = np.clip(rng.normal(-3.0, 4.0, n), -25, 3)
    systolic_bp = np.clip(rng.normal(125, 16, n), 90, 190)
    diastolic_bp = np.clip(rng.normal(80, 10, n), 55, 120)
    corneal_curvature_d = np.clip(rng.normal(43.5, 1.6, n), 38, 48)
    family_history_glaucoma = rng.binomial(1, 0.25, n)

    noise = rng.normal(0, 1.6, n)

    iop = (
        9.5
        + 0.055 * age
        + 0.018 * (corneal_thickness_um - 520)
        + 4.2 * cup_disc_ratio
        - 0.10 * visual_field_md_db
        + 0.018 * (systolic_bp - 120)
        + 0.55 * family_history_glaucoma
        + 0.10 * (corneal_curvature_d - 43)
        + noise
    )

    iop = np.clip(iop, 8, 35)

    df = pd.DataFrame({
        "age": age,
        "corneal_thickness_um": np.round(corneal_thickness_um, 1),
        "cup_disc_ratio": np.round(cup_disc_ratio, 2),
        "visual_field_md_db": np.round(visual_field_md_db, 2),
        "systolic_bp": np.round(systolic_bp, 1),
        "diastolic_bp": np.round(diastolic_bp, 1),
        "corneal_curvature_d": np.round(corneal_curvature_d, 2),
        "family_history_glaucoma": family_history_glaucoma,
        "intraocular_pressure_mmhg": np.round(iop, 2),
    })

df.head()

## 4. Check Dataset

In [ ]:
print("Dataset shape:", df.shape)
print("\nMissing values:")
print(df.isnull().sum())

df.describe().round(2)

## 5. Simple Visualization

In [ ]:
df["intraocular_pressure_mmhg"].hist(bins=20)
plt.xlabel("Intraocular Pressure (mmHg)")
plt.ylabel("Number of Patients")
plt.title("Distribution of Intraocular Pressure")
plt.show()

## 6. Select Features (X) and Target (y)

In [ ]:
X = df.drop("intraocular_pressure_mmhg", axis=1)
y = df["intraocular_pressure_mmhg"]

print("Input features:")
print(list(X.columns))

print("\nTarget:")
print(y.name)

## 7. Train/Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows :", X_test.shape[0])

## 8. Create and Train XGBoost Regressor

In [ ]:
model = XGBRegressor(
    n_estimators=200,
    max_depth=3,
    learning_rate=0.05,
    subsample=0.9,
    colsample_bytree=0.9,
    objective="reg:squarederror",
    random_state=42
)

model.fit(X_train, y_train)

print("XGBoost model trained successfully.")

## 9. Make Predictions

In [ ]:
y_pred = model.predict(X_test)

results = pd.DataFrame({
    "Actual_IOP": y_test.values,
    "Predicted_IOP": y_pred
})

results.head(10).round(2)

## 10. Evaluate the Regression Model

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE  : {mae:.3f} mmHg")
print(f"RMSE : {rmse:.3f} mmHg")
print(f"R²   : {r2:.3f}")

### How to understand the metrics

- **MAE:** average absolute prediction error. Lower is better.
- **RMSE:** gives more penalty to large errors. Lower is better.
- **R²:** how much variation in IOP is explained by the model. Closer to 1 is better.


## 11. Actual vs Predicted IOP

In [ ]:
plt.scatter(y_test, y_pred, alpha=0.7)
plt.xlabel("Actual IOP (mmHg)")
plt.ylabel("Predicted IOP (mmHg)")
plt.title("Actual vs Predicted IOP")

minimum = min(y_test.min(), y_pred.min())
maximum = max(y_test.max(), y_pred.max())
plt.plot([minimum, maximum], [minimum, maximum])

plt.show()

## 12. Feature Importance

In [ ]:
importance = pd.Series(
    model.feature_importances_,
    index=X.columns
).sort_values()

importance.plot(kind="barh")
plt.xlabel("Importance")
plt.title("XGBoost Feature Importance")
plt.show()

importance.sort_values(ascending=False)

## 13. Predict IOP for One New Example

In [ ]:
new_patient = pd.DataFrame({
    "age": [55],
    "corneal_thickness_um": [535],
    "cup_disc_ratio": [0.55],
    "visual_field_md_db": [-4.5],
    "systolic_bp": [135],
    "diastolic_bp": [85],
    "corneal_curvature_d": [43.8],
    "family_history_glaucoma": [1]
})

predicted_iop = model.predict(new_patient)[0]

print(f"Predicted IOP: {predicted_iop:.2f} mmHg")

## 14. Optional: Save the Trained Model

This is useful if you later want to connect the model to **Streamlit** or **FastAPI**.


In [ ]:
import joblib

joblib.dump(model, "xgboost_eye_iop_model.pkl")
print("Model saved as xgboost_eye_iop_model.pkl")

# Conclusion

In this notebook we:

1. Loaded an eye-related medical regression dataset.
2. Selected input features and an IOP target.
3. Split the dataset into training and testing data.
4. Trained an `XGBRegressor`.
5. Evaluated it using MAE, RMSE, and R².
6. Visualized actual vs predicted IOP.
7. Checked feature importance.
8. Predicted IOP for a new sample.

### Medical disclaimer
This notebook uses **synthetic educational data**. The model is not validated for clinical use and should not be used to diagnose glaucoma or make treatment decisions.
